# Part 2: Agentic EDA with an AI Data Science Team
**CMPE 255: exploratory data analysis done by AI agents instead of by hand**

| | |
|---|---|
| **Tool** | [business-science/ai-data-science-team](https://github.com/business-science/ai-data-science-team) (from the course slide deck): a library of specialised LangChain/LangGraph agents (EDA, cleaning, visualisation, wrangling, feature engineering, ML) that receive natural-language instructions, **choose tools or write pandas/plotly code themselves, execute it and explain the result**. |
| **Dataset** | `student_grade_ses_dataset.csv`: 12,025 students × 29 columns. Demographics, family socio-economic status (SES), study habits, attendance and grades. Targets: `final_grade_pct`, `grade_class`, `at_risk_flag`. |
| **Question** | *What drives student performance, and which students are at risk?* |
| **LLM** | Any tool-calling chat model. OpenAI `gpt-4o-mini` by default (the library's own default); a local Ollama model also works. |

### Why this tool
* It covers the **whole EDA workflow** with one agent per stage, mirroring a real data-science team (see the architecture on the slide).
* It is **transparent**: every agent returns the tools it called, or the Python code it wrote and ran, so each step can be inspected and verified.
* It runs in a plain notebook or Colab with a single `pip install`, so there's no platform account to set up.

### How the agents are used here
```
                    ┌───────────────────────── natural-language prompts ─────────────────────────┐
 raw CSV ──► ① EDA Tools Agent ──► ② Data Cleaning Agent ──► ③ EDA Tools Agent (on clean data)
             explain / describe /     writes + runs a            correlation funnel for at-risk
             missingno / Sweetviz     cleaning function
                                            │
                                            ├──► ④ Data Visualization Agent  (writes plotly code)
                                            ├──► ⑤ Pandas Data Analyst      (wrangling + viz sub-agents)
                                            └──► ⑥ Feature Engineering Agent (model-ready features)
                                                        │
                                      ⑦ Human verification ◄─┘   ⑧ Agent vs manual (Part 1) reflection
```

> ⚠️ **Safety note:** the cleaning, visualisation, wrangling and feature-engineering agents **execute Python code written by the LLM**. The library runs cleaning code in a restricted subprocess. All generated code is printed below so it can be reviewed, and section 7 independently re-checks the agents' claims.

### How this notebook was built
> ✍️ *TODO (you): explain how you used a coding agent to scaffold this notebook, and what you ran, checked and wrote yourself.*

## 0. Setup
On Colab the next cell installs the pinned packages from `part2/requirements.txt` and **restarts the runtime once**. Afterwards choose *Runtime → Run all*. Locally, `pip install -r part2/requirements.txt` in a **separate environment from Part 1**, because Part 1's AutoViML pins conflict with this library.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules


def _env_ready():
    try:
        import importlib.metadata as md
        return md.version("ai-data-science-team") == "0.0.0.9017"
    except Exception:
        return False


if IN_COLAB and not _env_ready():
    %pip install -q ai-data-science-team==0.0.0.9017 missingno==0.5.2 sweetviz==2.3.3 pytimetk==2.5.3
    print("Installed packages -> restarting runtime. Afterwards: Runtime -> Run all.")
    os.kill(os.getpid(), 9)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json, time, base64, html, io, contextlib, logging
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown, Image, HTML

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # silence font warnings from plotnine
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
if IN_COLAB:
    pio.renderers.default = "colab"

### 0.1 Workarounds for three library bugs
While testing this notebook I found three bugs in `ai-data-science-team` 0.0.0.9017 that crash agents in normal use. All are patched at runtime here; the library files themselves are not modified:
1. **Cleaning, visualisation and other agents crash whenever `user_instructions` is given.** They seed the conversation with a `("user", text)` tuple, and a post-processing helper (`remove_consecutive_duplicates`) then calls `.content` on it, raising `AttributeError`.
2. **The visualisation agent's error recovery crashes.** When LLM-written chart code fails, the agent tries to read the error message with regexes that are double-escaped (`r"\\['…"`). That raises `re.error` instead of retrying. The patch makes those lookups fail safely so the normal retry loop runs.
3. **A `print()` in LLM-written code makes the fix prompt explode.** The sandbox returns its result as JSON on stdout. Any printed text breaks parsing, and the library then pastes the *entire* stdout into the error message sent back to the LLM. That includes the full result table: about 5.9 million characters for this dataset. OpenAI rejects the request with a 429 *request too large* error. The patch recovers the real result from the last stdout line, and caps any error message at 4,000 characters.

In [ ]:
import re as _re, types as _types
import ai_data_science_team.templates.agent_templates as _tmpl
import ai_data_science_team.agents.data_visualization_agent as _viz


def _remove_consecutive_duplicates(messages):
    out, prev = [], object()
    for m in messages:
        content = m.content if hasattr(m, "content") else (m[1] if isinstance(m, tuple) and len(m) == 2 else m)
        if content != prev:
            out.append(m)
        prev = content
    return out


class _SafeRe(_types.SimpleNamespace):
    def findall(self, *a, **k):
        try:
            return _re.findall(*a, **k)
        except _re.error:
            return []

    def search(self, *a, **k):
        try:
            return _re.search(*a, **k)
        except _re.error:
            return None


_tmpl.remove_consecutive_duplicates = _remove_consecutive_duplicates
_viz.re = _SafeRe(**{k: getattr(_re, k) for k in dir(_re) if not k.startswith("_") and k not in ("findall", "search")})

# Bug 3: if LLM-written code calls print(), the sandbox's stdout is no longer pure JSON. The library
# then puts the *whole* stdout, including the full result table, into the error message that is
# sent back to the LLM for fixing (millions of tokens -> OpenAI 429 "request too large").
import ast as _ast, json as _json, importlib as _importlib
from ai_data_science_team.utils import sandbox as _sandbox

_NON_JSON = "Sandbox returned non-JSON output: "
_MAX_ERR_CHARS = 4000
_orig_sandbox = _sandbox.run_code_sandboxed_subprocess


def _shorten(msg, limit=_MAX_ERR_CHARS):
    if msg is None or len(msg) <= limit:
        return msg
    half = limit // 2
    return f"{msg[:half]}\n... [{len(msg) - limit:,} characters truncated] ...\n{msg[-half:]}"


def _run_code_sandboxed_subprocess(**kwargs):
    result, error = _orig_sandbox(**kwargs)
    if error and error.startswith(_NON_JSON):
        try:  # recover the real result: the runner's JSON is the last line of stdout
            stdout = _ast.literal_eval(error[len(_NON_JSON):])
            parsed = _json.loads(stdout.rstrip().rsplit("\n", 1)[-1])
            result, error = parsed.get("result"), parsed.get("error")
        except Exception:
            error = _NON_JSON + "(the code printed to stdout, which breaks the sandbox; remove print() calls)"
    return result, _shorten(error)


for _mod in ("data_cleaning_agent", "data_wrangling_agent", "data_visualization_agent", "feature_engineering_agent"):
    _importlib.import_module(f"ai_data_science_team.agents.{_mod}").run_code_sandboxed_subprocess = _run_code_sandboxed_subprocess

print("Patched: remove_consecutive_duplicates, data_visualization_agent.re, sandbox output handling")

### 0.2 Choose the LLM and provide the API key
* **`openai`** (default): needs an `OPENAI_API_KEY`. On Colab, add it under 🔑 *Secrets* with the name `OPENAI_API_KEY` and enable notebook access. Locally, export it as an environment variable. **Never paste the key into a cell**, because executed notebooks get committed to GitHub.
* **`ollama`**: free and local. Needs [Ollama](https://ollama.com) running with a tool-calling model pulled (e.g. `ollama pull qwen2.5:14b`). Small local models make more tool-calling mistakes.

*Why not Claude?* This library version reads replies as plain strings (e.g. `.content.strip()`). Current Claude models return a list of thinking and text blocks, so several agents would crash without patching the library.

If no key is found, `RUN_AGENTS` becomes `False` and the agent cells are skipped. The rest of the notebook (data load, tool smoke test, human verification) still runs.

In [ ]:
LLM_PROVIDER = "openai"          # "openai" or "ollama"
MODEL_NAME = "gpt-4o-mini"       # library default; any tool-calling chat model on your account works
OLLAMA_MODEL = "qwen2.5:14b"


def _load_openai_key():
    if os.environ.get("OPENAI_API_KEY"):
        return True
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
            return True
        except Exception as e:
            print("Colab secret OPENAI_API_KEY not available:", type(e).__name__)
    return False


llm = None
if LLM_PROVIDER == "openai" and _load_openai_key():
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL_NAME)
elif LLM_PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL)

RUN_AGENTS = llm is not None
print(f"Provider: {LLM_PROVIDER} | model: {getattr(llm, 'model_name', None) or getattr(llm, 'model', None)} | RUN_AGENTS = {RUN_AGENTS}")

### 0.3 Load the data

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/anita-agasaveeran/CMPE255_Exploratory_Data_Analysis/main/data/"
DATA_FILE = "student_grade_ses_dataset.csv"


def load_csv(name):
    for base in [Path("../data"), Path("data"), Path("/content/data")]:
        if (base / name).exists():
            print(f"Loading local file {base / name}")
            return pd.read_csv(base / name)
    print(f"Loading {REPO_RAW + name}")
    return pd.read_csv(REPO_RAW + name)


df_raw = load_csv(DATA_FILE)
print(df_raw.shape)
df_raw.head(3)

That is the only manual look at the data. Everything below is done by the agents.

### 0.4 Helpers: run an agent, show what it did, log it
Each call records the prompt, the tools or code the agent used, the runtime and, for OpenAI, the token cost. Section 8 summarises this log, which makes the agents' work auditable.

In [ ]:
try:
    from langchain_community.callbacks import get_openai_callback
except Exception:
    get_openai_callback = None

AGENT_LOG = []
SHOW_SWEETVIZ_INLINE = True


@contextlib.contextmanager
def track(agent_name, prompt):
    entry = {"agent": agent_name, "prompt": prompt[:90] + ("…" if len(prompt) > 90 else ""),
             "seconds": None, "tokens": None, "cost_usd": None, "status": "ok"}
    t0 = time.time()
    cb_ctx = get_openai_callback() if (get_openai_callback and LLM_PROVIDER == "openai") else contextlib.nullcontext()
    try:
        with cb_ctx as cb:
            yield entry
        if cb is not None:
            entry["tokens"], entry["cost_usd"] = cb.total_tokens, round(cb.total_cost, 4)
    except Exception as e:
        entry["status"] = f"error: {type(e).__name__}: {str(e)[:120]}"
        display(Markdown(f"> ❌ **Agent error:** `{type(e).__name__}: {str(e)[:300]}`"))
    finally:
        entry["seconds"] = round(time.time() - t0, 1)
        AGENT_LOG.append(entry)


def show_md(x):
    if x is None:
        return
    display(x if hasattr(x, "_repr_markdown_") else Markdown(str(x)))


def _is_png_b64(s):
    return isinstance(s, str) and s.startswith("iVBORw0KGgo")


def render_artifact(obj, label=""):
    """Render whatever an EDA tool returned: images, tables, plotly figures, reports."""
    if obj is None:
        return
    if _is_png_b64(obj):
        if label:
            display(Markdown(f"**{label}**"))
        display(Image(data=base64.b64decode(obj)))
    elif isinstance(obj, dict):
        if "data" in obj and "layout" in obj:                                   # plotly figure
            pio.from_json(json.dumps(obj)).show()
        elif "describe_df" in obj:                                              # describe_dataset
            display(pd.DataFrame(obj["describe_df"]).set_index("stat").T)
        elif "report_file" in obj:                                              # sweetviz
            display(Markdown(f"📄 Sweetviz report saved to `{obj['report_file']}`"))
            if SHOW_SWEETVIZ_INLINE and obj.get("report_html"):
                display(HTML(f'<iframe srcdoc="{html.escape(obj["report_html"])}" width="100%" height="750" style="border:1px solid #ccc"></iframe>'))
        elif "correlation_data" in obj:                                         # correlation funnel
            render_artifact(obj.get("plot_image"), "Correlation funnel")
            corr = pd.DataFrame(obj["correlation_data"])
            display(corr.reindex(corr["correlation"].abs().sort_values(ascending=False).index).head(15))
        else:
            for k, v in obj.items():
                render_artifact(v, k)


def ask_eda(prompt, data):
    """Send a natural-language EDA request to the EDA Tools Agent and show its reasoning trail."""
    display(Markdown(f"### 🧑 Prompt\n> {prompt}"))
    with track("EDA Tools Agent", prompt):
        eda_agent.invoke_agent(user_instructions=prompt, data_raw=data)
        display(Markdown(f"**🔧 Tools the agent chose:** `{eda_agent.get_tool_calls()}`"))
        display(Markdown("**🤖 Agent's answer:**"))
        show_md(eda_agent.get_ai_message(markdown=True))
        render_artifact(eda_agent.get_artifacts())

### 0.5 Tool smoke test (no LLM needed)
Before handing the tools to an agent, check that they work in this environment by calling two of them directly on the raw data. This shows exactly what the agent will receive when it calls them.

In [ ]:
import matplotlib.pyplot as plt
from ai_data_science_team.tools.eda import describe_dataset, visualize_missing
with contextlib.redirect_stdout(io.StringIO()):
    _, art_desc = describe_dataset.func(data_raw=df_raw.to_dict())
    _, art_miss = visualize_missing.func(data_raw=df_raw.to_dict(), n_sample=2000)
plt.close("all")   # the tool leaves empty figures open; it returns its plots as base64 images
print("describe_dataset →", list(art_desc), "| visualize_missing →", list(art_miss))
render_artifact(art_miss.get("bar_plot"), "missingno bar plot (direct tool call)")

## 1. EDA Tools Agent on the raw data
The **EDA Tools Agent** is a ReAct agent. It reads each request, **decides which EDA tool to call** (`explain_data`, `describe_dataset`, `visualize_missing`, `generate_correlation_funnel`, `generate_sweetviz_report`, …), runs it and then explains the results.

In [ ]:
if RUN_AGENTS:
    from ai_data_science_team.ds_agents import EDAToolsAgent
    eda_agent = EDAToolsAgent(model=llm)

In [ ]:
if RUN_AGENTS:
    ask_eda("Give me a thorough overview of this dataset: what each column represents, the data types, "
            "and anything that looks like a data-quality problem.", df_raw)

In [ ]:
if RUN_AGENTS:
    ask_eda("Compute descriptive statistics for every column and point out skewed variables or suspicious ranges.", df_raw)

In [ ]:
if RUN_AGENTS:
    ask_eda("Visualize the missing values. Which columns have missing data, and does the missingness look "
            "random or related between columns?", df_raw)

In [ ]:
if RUN_AGENTS:
    ask_eda("Generate a Sweetviz report with final_grade_pct as the target and summarise its key findings.", df_raw)

> ✍️ **Your observations (agent EDA on raw data):** did the agent pick sensible tools? Did it notice the 25 duplicate rows, the ~6% missing income, the skewed income distribution and the tiny at-risk class (~1.8%)? Note anything it got wrong or invented.

## 2. Data Cleaning Agent
The **Data Cleaning Agent** (1) looks at a sample of the data, (2) writes a list of recommended cleaning steps, (3) **writes a Python `data_cleaner()` function**, (4) runs it in a sandbox and retries with the error message if it fails, and (5) returns the cleaned data. Specific instructions are given so it doesn't apply blind defaults.

In [ ]:
CLEANING_INSTRUCTIONS = """
Clean this student dataset for analysis:
- Remove exact duplicate rows.
- Do NOT drop rows just because they have missing values.
- Impute numeric columns with the median and categorical columns with the most frequent value.
- Keep the target columns final_grade_pct, grade_class and at_risk_flag unchanged.
- Convert survey_date to a datetime.
- Do not remove outliers in annual_family_income_inr (high incomes are real), but cap study_hours_per_week at its 99th percentile.
"""

if RUN_AGENTS:
    from ai_data_science_team.agents import DataCleaningAgent
    cleaning_agent = DataCleaningAgent(model=llm, n_samples=50, log=True, log_path="agent_logs")
    display(Markdown(f"### 🧑 Prompt\n```\n{CLEANING_INSTRUCTIONS.strip()}\n```"))
    with track("Data Cleaning Agent", CLEANING_INSTRUCTIONS.strip()):
        cleaning_agent.invoke_agent(data_raw=df_raw, user_instructions=CLEANING_INSTRUCTIONS)
        display(Markdown("**🤖 Recommended cleaning steps:**"))
        show_md(cleaning_agent.get_recommended_cleaning_steps(markdown=True))
        display(Markdown("**🐍 Code the agent wrote and executed:**"))
        show_md(cleaning_agent.get_data_cleaner_function(markdown=True))
    df_clean = cleaning_agent.get_data_cleaned()
else:
    df_clean = None

In [ ]:
def quality_snapshot(d, name):
    return pd.Series({"rows": len(d), "duplicate rows": d.duplicated().sum(),
                      "cells missing": int(d.isna().sum().sum()),
                      "columns with missing": int((d.isna().sum() > 0).sum())}, name=name)


snap = [quality_snapshot(df_raw, "raw")]
if df_clean is not None:
    snap.append(quality_snapshot(df_clean, "cleaned by agent"))
pd.concat(snap, axis=1)

> ✍️ **Your observations (cleaning agent):** did the generated code follow every instruction? Read it line by line. Is anything risky, such as imputing before deduplicating or touching the targets?

## 3. EDA Tools Agent on the cleaned data: what predicts being at risk?
The **correlation funnel** tool binarises every feature and ranks how strongly each bin correlates with a target. It **fails on data with missing values**, which was verified in the tool test, so it only works after the cleaning agent has run. This is a natural example of agents depending on each other.

In [ ]:
if RUN_AGENTS and df_clean is not None:
    funnel_data = df_clean.drop(columns=[c for c in ["student_id", "survey_date"] if c in df_clean.columns])
    ask_eda("Build a correlation funnel with at_risk_flag as the target (target level 1). "
            "Which features and value ranges are most associated with a student being at risk?", funnel_data)

## 4. Data Visualization Agent
Instead of calling pre-built tools, this agent **writes plotly code** for each request, executes it and returns the figure and the code.

In [ ]:
VIZ_PROMPTS = [
    "Box plot of final_grade_pct by family_income_bracket, with brackets ordered from Low to High.",
    "Scatter plot of attendance_rate_pct vs final_grade_pct coloured by at_risk_flag, with a trendline.",
    "Grouped bar chart of average final_grade_pct by has_internet_access and residence_type.",
]

if RUN_AGENTS and df_clean is not None:
    from ai_data_science_team.agents import DataVisualizationAgent
    viz_agent = DataVisualizationAgent(model=llm, n_samples=30, log=True, log_path="agent_logs")
    for prompt in VIZ_PROMPTS:
        display(Markdown(f"### 🧑 Prompt\n> {prompt}"))
        with track("Data Visualization Agent", prompt):
            viz_agent.invoke_agent(data_raw=df_clean, user_instructions=prompt)
            fig = viz_agent.get_plotly_graph()
            if fig is not None:
                fig.show()
            with contextlib.suppress(Exception):
                show_md(Markdown(f"```python\n{viz_agent.response.get('data_visualization_function', '')}\n```"))

> ✍️ **Your observations (visualisation agent):** were the charts correct and readable? Did it handle the bracket ordering? How does this compare with writing the matplotlib/seaborn code yourself in Part 1?

## 5. Pandas Data Analyst (multi-agent)
The **Pandas Data Analyst** is a small multi-agent system. A router decides whether a question needs a **wrangling** sub-agent (which writes pandas code for groupby, pivot or filter), a **visualisation** sub-agent, or both. These are free-form analytical questions, the kind an analyst would explore by hand.

In [ ]:
ANALYST_QUESTIONS = [
    "What is the average final_grade_pct and the at-risk rate for each school_type and family_income_bracket combination? Show a table.",
    "Does extra_tutoring help? Compare mean final_grade_pct with and without tutoring within each family_income_bracket and plot it.",
    "Which 5 numeric features have the strongest correlation with final_grade_pct? Show them in a bar chart.",
]

if RUN_AGENTS and df_clean is not None:
    from ai_data_science_team.agents import DataWranglingAgent, DataVisualizationAgent
    from ai_data_science_team.multiagents import PandasDataAnalyst
    analyst = PandasDataAnalyst(model=llm,
                                data_wrangling_agent=DataWranglingAgent(model=llm, log=True, log_path="agent_logs"),
                                data_visualization_agent=DataVisualizationAgent(model=llm, log=True, log_path="agent_logs"))
    for q in ANALYST_QUESTIONS:
        display(Markdown(f"### 🧑 Question\n> {q}"))
        with track("Pandas Data Analyst", q):
            analyst.invoke_agent(user_instructions=q, data_raw=df_clean)
            with contextlib.suppress(Exception):
                display(analyst.get_data_wrangled().head(20))
            fig = analyst.get_plotly_graph()
            if fig is not None:
                fig.show()
            show_md(analyst.get_data_wrangler_function(markdown=True))

## 6. Feature Engineering Agent
This agent prepares a **model-ready** table for `final_grade_pct`: encoding, handling high-cardinality and ID columns, and dropping constant columns. The aim is to see what an agent would hand to an ML step, as a comparison to the manual feature engineering in Part 1.

In [ ]:
FE_INSTRUCTIONS = ("Prepare features to predict final_grade_pct. Drop student_id and survey_date, do not use grade_class or "
                   "at_risk_flag as features (they are derived from the target), one-hot encode categorical columns and "
                   "create a total_parent_education feature from the two parent education levels.")

if RUN_AGENTS and df_clean is not None:
    from ai_data_science_team.agents import FeatureEngineeringAgent
    fe_agent = FeatureEngineeringAgent(model=llm, log=True, log_path="agent_logs")
    display(Markdown(f"### 🧑 Prompt\n> {FE_INSTRUCTIONS}"))
    with track("Feature Engineering Agent", FE_INSTRUCTIONS):
        fe_agent.invoke_agent(data_raw=df_clean, user_instructions=FE_INSTRUCTIONS, target_variable="final_grade_pct")
        df_features = fe_agent.get_data_engineered()
        print("Engineered table:", df_features.shape)
        display(df_features.head(3))
        show_md(fe_agent.get_feature_engineer_function(markdown=True))

> ✍️ **Your observations (feature engineering):** did the agent respect the leakage instruction? Compare with the leakage checks you did manually in Part 1.

## 7. Human verification: trust, but verify
LLM agents can sound confident and still be wrong. The cell below recomputes key facts **independently with plain pandas**, so they can be compared with what the agents reported in sections 1–5. Any disagreement goes in the reflection.

In [ ]:
facts = {
    "rows": len(df_raw),
    "exact duplicate rows": int(df_raw.duplicated().sum()),
    "columns with missing values": int((df_raw.isna().sum() > 0).sum()),
    "most-missing column": f"{df_raw.isna().sum().idxmax()} ({df_raw.isna().mean().max():.1%})",
    "at-risk rate": f"{df_raw['at_risk_flag'].mean():.2%}",
    "mean final_grade_pct": round(df_raw["final_grade_pct"].mean(), 2),
    "corr(attendance, final grade)": round(df_raw[["attendance_rate_pct", "final_grade_pct"]].corr().iloc[0, 1], 3),
    "corr(study hours, final grade)": round(df_raw[["study_hours_per_week", "final_grade_pct"]].corr().iloc[0, 1], 3),
}
display(pd.Series(facts, name="ground truth (pandas)").to_frame())

order = ["Low", "Lower-Middle", "Middle", "Upper-Middle", "High"]
display(df_raw.groupby("family_income_bracket")[["final_grade_pct", "at_risk_flag"]].mean()
        .reindex(order).round(3).rename(columns={"at_risk_flag": "at_risk_rate"}))

## 8. Agent run log

In [ ]:
log_df = pd.DataFrame(AGENT_LOG)
if len(log_df):
    display(log_df)
    print(f"Total agent time: {log_df['seconds'].sum():.0f}s"
          + (f" | total cost: ${log_df['cost_usd'].sum():.3f}" if log_df["cost_usd"].notna().any() else ""))
else:
    print("No agent calls were made (RUN_AGENTS is False). Set an API key and re-run.")

## 9. Reflection: agentic EDA vs manual EDA (Part 1)
> ✍️ **TODO (you), after running everything:**

| Aspect | Manual (Part 1) | Agents (Part 2) |
|---|---|---|
| Time to first insights | | |
| Coverage (what was / wasn't checked) | | |
| Correctness (errors or hallucinations found in section 7) | | |
| Leakage awareness | | |
| Transparency and reproducibility | | |
| Cost | free | see section 8 |

**Key insights about student performance:** …

**When I would (and wouldn't) trust an EDA agent:** …